# Hafta 3 — Kaggle CLIP Embedding Cikarma

**Amac:** 17.176 laundered goruntu icin dondurulmus CLIP ViT-L/14 embedding'i
cikarip `.npy` olarak indirmek. Sonrasindaki her sey (grid search, esik secimi,
kalibrasyon, E3/E5/E6) yerelde **dakikalar** icinde doner.

## Neden goruntuleri yuklemiyoruz

Laundered goruntuler ~5-6 GB. Bunun yerine Kaggle'da **yeniden uretiyoruz**:
`cardd-data` zaten Kaggle'da duruyor, uretilmis S/M katmanlari 585 MB'lik iki
zip. Laundering yerelde 90 saniye surmustu -- Kaggle'da da hizli.

Uretilen manifest'in yereldekiyle AYNI oldugu, **dondurulmus test setinin
sha256'si** ile dogrulanir (hucre 4). Tutmazsa embedding'ler hizalanmaz ve
durmamiz gerekir.

## Onkosullar (calistirmadan once)

1. **Accelerator:** GPU T4 x2 (tek T4 de yeter, bu is tek GPU kullanir)
2. **Input olarak eklenmis 2 dataset:**
   - `cardd-data` (W2'den, zaten ekli olmali)
   - **YENI:** `w2-uretim` — `w2_synthetic.zip` + `w2_manipulated.zip`
     dosyalarini iceren bir dataset. Bunlari bilgisayarindan yukle:
     *Add Input -> Upload -> Create Dataset*
3. Internet: **acik** (repo klonlama + CLIP agirliklari icin)


## 0. Ortam ve GPU

In [ ]:
import os, sys, platform, subprocess, torch
from pathlib import Path

print("Python :", platform.python_version())
print("torch  :", torch.__version__)
print("GPU sayisi:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB")

import shutil
print("\nDisk (/kaggle/working):", f"{shutil.disk_usage('/kaggle/working').free/1e9:.0f} GB bos")

WORK = Path("/kaggle/working/insurance-image-forensics")
INPUT = Path("/kaggle/input")
print("\nInput datasetleri:")
for d in sorted(INPUT.iterdir()):
    print("  ", d.name)

In [ ]:
!pip install -q transformers accelerate safetensors pyarrow 2>&1 | tail -3
import transformers, sklearn
print("transformers", transformers.__version__)
print("sklearn", sklearn.__version__)

## 1. Repo, veri ve uretim ciktilari

In [ ]:
REPO_URL = "https://github.com/Tunahan-46/insurance-image-forensics.git"

if not WORK.exists():
    !git clone -q $REPO_URL {str(WORK)}
else:
    print("Repo zaten var, guncelleniyor")
    !cd {str(WORK)} && git pull -q

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd())
!git log --oneline -1

In [ ]:
# CarDD'yi kodun bekledigi yere BAGLA (kopyalama yok)
DATA = next(p for p in INPUT.iterdir() if "cardd" in p.name.lower())
CARDD = WORK / "data/raw/cardd"
CARDD.mkdir(parents=True, exist_ok=True)

coco_src = next(DATA.rglob("CarDD_COCO"))
if not (CARDD / "CarDD_COCO").exists():
    os.symlink(coco_src, CARDD / "CarDD_COCO")

SOD = CARDD / "CarDD_SOD"
for split in ["TR", "VAL", "TE"]:
    dst_dir = SOD / f"CarDD-{split}"
    dst = dst_dir / f"CarDD-{split}-Mask"
    if dst.exists():
        continue
    hits = list(DATA.rglob(f"CarDD-{split}-Mask"))
    if not hits:
        print(f"  UYARI: CarDD-{split}-Mask bulunamadi")
        continue
    dst_dir.mkdir(parents=True, exist_ok=True)
    os.symlink(hits[0], dst)

for p in ["CarDD_COCO/train2017", "CarDD_COCO/val2017", "CarDD_COCO/test2017",
          "CarDD_SOD/CarDD-TR/CarDD-TR-Mask", "CarDD_SOD/CarDD-VAL/CarDD-VAL-Mask",
          "CarDD_SOD/CarDD-TE/CarDD-TE-Mask"]:
    d = CARDD / p
    n = len(list(d.iterdir())) if d.exists() else "YOK"
    print(f"  {p:<42} {n}")

In [ ]:
# W2'de uretilen S / M1 / M2 (+ yerel M3 = classic) katmanlarini bagla.
#
# NEDEN HEM ZIP HEM KLASOR DESTEKLENIYOR
# ---------------------------------------
# Kaggle "New Dataset" akisi bazen yuklenen .zip'i OTOMATIK ACAR ve dataset
# icinde zip'in adiyla ayni isimde bir KLASOR birakir (zip dosyasinin
# kendisi degil). Hangisinin olacagi yukleme yontemine gore degisir; bu
# yuzden ikisini de arayip bulani kullaniyoruz -- sessizce atlamak yerine.
#
# YAPISAL KURAL: hedef klasor (data/raw/...) her zaman GERCEK bir dizin
# olarak yaratilir, sonra SADECE ICINDEKI alt klasorler sembolik baglanir
# (bkz. hucre 6'daki CarDD ayni deseni kullaniyor). Hedefin KENDISINI
# Kaggle'in salt-okunur input'una baglarsak, sonradan "classic" alt
# klasorunu data/raw/manipulated icine EKLEYEMEYIZ -- salt-okunur olur.

import os
import zipfile


def find_zip(pattern: str):
    for d in INPUT.iterdir():
        for z in d.rglob(pattern):
            return z
    return None


def find_dir(name: str):
    for d in INPUT.iterdir():
        for sub in d.rglob(name):
            if sub.is_dir():
                return sub
    return None


targets = {
    "w2_synthetic": "data/raw/synthetic",
    "w2_manipulated": "data/raw/manipulated",
    "w2_classic": "data/raw/manipulated/classic",
}

found_any = False
for base, rel in targets.items():
    dst = WORK / rel

    zpath = find_zip(f"{base}.zip")
    if zpath is not None:
        found_any = True
        dst.mkdir(parents=True, exist_ok=True)
        if not any(dst.iterdir()):
            with zipfile.ZipFile(zpath) as z:
                z.extractall(dst)
        print(f"  {rel}: zip'ten acildi ({len(list(dst.rglob('*.png')))} png)")
        continue

    dpath = find_dir(base)
    if dpath is not None:
        found_any = True
        dst.mkdir(parents=True, exist_ok=True)
        for child in dpath.iterdir():
            link = dst / child.name
            if not link.exists():
                os.symlink(child, link)
        print(f"  {rel}: Kaggle onceden acmis, alt klasorler baglandi ({len(list(dst.rglob('*.png')))} png)")
        continue

    print(f"  UYARI: {base} (zip veya klasor) bulunamadi, atlandi")

assert found_any, (
    "Ne w2_synthetic ne w2_manipulated bulunamadi (zip ya da klasor olarak). "
    "Notebook basindaki onkosullara bak -- Input'a dogru dataset eklenmemis."
)

# M3 (klasik manipulasyon) yerelde uretilmisti ve W2 zip'lerine GIRMEMISTI.
# Burada yoksa manifest 520 satir eksik cikar ve sha256 TUTMAZ.
m3 = WORK / "data/raw/manipulated/classic"
n_m3 = len(list(m3.rglob("*.png"))) if m3.exists() else 0
print(f"\nM3 (classic): {n_m3} png")
if n_m3 == 0:
    print("  >>> M3 YOK. w2_classic.zip'i (veya acilmis w2_classic/ klasorunu)")
    print("  >>> Input'a ekle: scripts/make_classic_zip.py ile uret, Kaggle'a yukle.")


## 2. Manifest + laundering

Bu iki adim yerelde de calistirildi. Burada **ayni ciktiyi** uretmeleri
gerekiyor -- split'ler id hash'inden deterministik turetiliyor.

In [ ]:
!python scripts/build_manifest_v2.py

In [ ]:
!python scripts/apply_laundering.py

### 3b. BUTUNLUK KAPISI — test seti sha256

Yereldeki dondurulmus test setiyle **birebir ayni** manifest uretildi mi?
Tutmuyorsa embedding'ler yerel manifest'e hizalanmaz; **devam etme.**

In [ ]:
YEREL_SHA = "f46588e9b078a43d9771f5f7e77b0a83841f34d76a76307b177c6d63b570485d"

!python scripts/build_manifest_v2.py --freeze-test 2>&1 | tail -6

from pathlib import Path
sha_path = Path("data/processed/test_manifest_frozen.sha256")
kaggle_sha = sha_path.read_text().strip() if sha_path.exists() else "(yok)"

print("\nyerel  :", YEREL_SHA)
print("kaggle :", kaggle_sha)
if kaggle_sha == YEREL_SHA:
    print("\n>>> TUTUYOR. Devam edebilirsin.")
else:
    print("\n>>> TUTMUYOR! Manifest farkli uretilmis.")
    print(">>> Muhtemel sebep: M3 (classic) katmani burada yok,")
    print(">>> ya da bir katmanin dosya sayisi farkli. DEVAM ETME,")
    print(">>> once sebebi bul.")

## 4. CLIP embedding cikarma

Dondurulmus CLIP ViT-L/14, 768-d. Parcalar halinde yazilir: oturum koparsa
(W2'de oldugu gibi) kalinan yerden devam eder.

In [ ]:
import subprocess

Path("logs").mkdir(exist_ok=True)

CMD = (
    "python -m src.features.clip_embed "
    "--manifest data/processed/manifest_v2_laundered.parquet "
    "--out data/processed/clip_cache "
    "--batch 64 --device cuda"
)

PROC = globals().get("PROC")
if PROC is not None and PROC.poll() is None:
    print(f"Zaten calisiyor (pid={PROC.pid}), tekrar baslatilmadi.")
else:
    log = open("logs/clip.log", "a")
    PROC = subprocess.Popen(CMD.split(), stdout=log, stderr=subprocess.STDOUT, cwd=str(WORK))
    print(f"Baslatildi (pid={PROC.pid})")
    print("Ilerlemeyi asagidaki hucreyle izle. T4'te ~20-30 dk bekleniyor.")

### 4b. Ilerleme izleme

In [ ]:
import time
from pathlib import Path

def durum():
    print("=" * 68)
    log = Path("logs/clip.log")
    if log.exists():
        satirlar = [s for s in log.read_text(errors="ignore").splitlines() if s.strip()]
        for s in satirlar[-6:]:
            print("  " + s[:108])
    cache = Path("data/processed/clip_cache")
    shards = sorted(cache.glob("shard_*.npy")) if cache.exists() else []
    print(f"\n  yazilan parca : {len(shards)}")
    if shards:
        import numpy as np
        toplam = sum(np.load(s, mmap_mode="r").shape[0] for s in shards)
        print(f"  embedding     : {toplam} / 17176  ({100*toplam/17176:.1f}%)")
    print()
    !nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader

durum()

## 5. Ciktilari indirilebilir hale getir

In [ ]:
import shutil, numpy as np
from pathlib import Path

cache = Path("data/processed/clip_cache")
shards = sorted(cache.glob("shard_*.npy"))
toplam = sum(np.load(s, mmap_mode="r").shape[0] for s in shards)
print(f"{len(shards)} parca, {toplam} embedding")

if toplam < 17176:
    print(f"\n!!! EKSIK: {17176 - toplam} embedding daha bekleniyor.")
    print("!!! Cikarma bitmeden zip alma -- once 4b ile bekle.")
else:
    hedef = "/kaggle/working/w3_clip_cache"
    shutil.make_archive(hedef, "zip", cache)
    mb = Path(hedef + ".zip").stat().st_size / 1e6
    print(f"\n{hedef}.zip  ({mb:.0f} MB)")
    print("\nSonraki adim:")
    print("  1. Save Version -> Quick Save -> 'Save output for this version'")
    print("  2. Output'tan w3_clip_cache.zip indir")
    print("  3. YERELDE: data/processed/clip_cache/ altina ac")
    print("  4. python -m src.detectors.clip_probe --task A")
    print("     python -m src.detectors.clip_probe --task B")